In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import tensorflow as tf
if tf.test.gpu_device_name():
    print('Default GPU Device: {}'.format(tf.test.gpu_device_name()))
    !nvidia-smi
else:
    print("Please install GPU version of TF")

Default GPU Device: /device:GPU:0
Thu Mar  5 05:37:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P0             28W /   70W |     105MiB /  15360MiB |      4%      Default |
|                                         |                        |                  N/A |
+-------------

In [3]:
import os

save_path = "/content/drive/MyDrive/BCI_Processed"
os.makedirs(save_path, exist_ok=True)

In [4]:
import numpy as np

t0 = np.load(f"{save_path}/t0.npy")
t1 = np.load(f"{save_path}/t1.npy")
t2 = np.load(f"{save_path}/t2.npy")

print("T0 shape:", t0.shape)
print("T1 shape:", t1.shape)
print("T2 shape:", t2.shape)

T0 shape: (1687, 64, 321)
T1 shape: (840, 64, 321)
T2 shape: (847, 64, 321)


In [ ]:
y0 = np.zeros(len(t0))
y1 = np.ones(len(t1))
y2 = np.ones(len(t2)) * 2

In [ ]:
X = np.concatenate([t0, t1, t2], axis=0)
y = np.concatenate([y0, y1, y2], axis=0)

In [ ]:
!pip install mne

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 28.2 MB/s eta 0:00:00


In [5]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import copy

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from torch.utils.data import TensorDataset, DataLoader

In [ ]:
# spliting data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (2699, 64, 321)
Test shape: (675, 64, 321)


In [ ]:
mean = X_train.mean(axis=(0,2), keepdims=True)
std = X_train.std(axis=(0,2), keepdims=True)

X_train = (X_train - mean) / (std + 1e-6)
X_test  = (X_test - mean) / (std + 1e-6)

In [ ]:
from mne.decoding import CSP

csp = CSP(n_components=32, log=True)

X_train_csp = csp.fit_transform(X_train, y_train)
X_test_csp = csp.transform(X_test)

print(X_train_csp.shape)

Computing rank from data with rank=None
    Using tolerance 80 (2.2e-16 eps * 64 dim * 5.6e+15  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating class=0.0 covariance using EMPIRICAL
Done.
Estimating class=1.0 covariance using EMPIRICAL
Done.
Estimating class=2.0 covariance using EMPIRICAL
Done.
(2699, 32)


In [ ]:
X_train_csp = X_train_csp[:, None, :, None]
X_test_csp = X_test_csp[:, None, :, None]

In [ ]:
train_dataset = TensorDataset(
    torch.tensor(X_train_csp, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long)
)

test_dataset = TensorDataset(
    torch.tensor(X_test_csp, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.long)
)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [ ]:
class CSPClassifier(nn.Module):

    def __init__(self, input_size=32, n_classes=3):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, n_classes)
        )

    def forward(self, x):

        x = x.view(x.size(0), -1)   # ⭐ fix shape

        return self.net(x)

In [ ]:
# class EEGNetCSP(nn.Module):

#     def __init__(self, n_channels=16, n_classes=3):
#         super().__init__()

#         self.conv = nn.Conv2d(1, 32, (n_channels,1))
#         self.relu = nn.ReLU()
#         self.fc = nn.Linear(32, n_classes)

#     def forward(self,x):

#         x = self.conv(x)
#         x = self.relu(x)

#         x = torch.flatten(x,1)

#         return self.fc(x)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cpu


In [ ]:
# model = EEGNetCSP(
#     n_channels=16,
#     n_classes=3
# ).to(device)
model = CSPClassifier(
    input_size=32,
    n_classes=3
).to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
epochs = 50
best_acc = 0

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for xb, yb in train_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        outputs = model(xb)

        loss = criterion(outputs, yb)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for xb, yb in test_loader:

            xb = xb.to(device)
            yb = yb.to(device)

            outputs = model(xb)

            _, preds = torch.max(outputs, 1)

            total += yb.size(0)
            correct += (preds == yb).sum().item()

    acc = correct / total

    print(f"Epoch {epoch+1} | Loss: {total_loss:.4f} | Accuracy: {acc:.4f}")

    if acc > best_acc:
        best_acc = acc

Epoch 1 | Loss: 44.2376 | Accuracy: 0.5007
Epoch 2 | Loss: 41.6124 | Accuracy: 0.5185
Epoch 3 | Loss: 39.8948 | Accuracy: 0.5170
Epoch 4 | Loss: 38.9657 | Accuracy: 0.5230
Epoch 5 | Loss: 37.9813 | Accuracy: 0.5259
Epoch 6 | Loss: 38.2049 | Accuracy: 0.5422
Epoch 7 | Loss: 37.3041 | Accuracy: 0.5378
Epoch 8 | Loss: 36.8412 | Accuracy: 0.5363
Epoch 9 | Loss: 36.6676 | Accuracy: 0.5378
Epoch 10 | Loss: 36.4983 | Accuracy: 0.5452
Epoch 11 | Loss: 36.1877 | Accuracy: 0.5422
Epoch 12 | Loss: 35.7179 | Accuracy: 0.5526
Epoch 13 | Loss: 35.6160 | Accuracy: 0.5511
Epoch 14 | Loss: 35.4637 | Accuracy: 0.5630
Epoch 15 | Loss: 35.2341 | Accuracy: 0.5600
Epoch 16 | Loss: 34.8020 | Accuracy: 0.5704
Epoch 17 | Loss: 34.5001 | Accuracy: 0.5748
Epoch 18 | Loss: 34.6300 | Accuracy: 0.5793
Epoch 19 | Loss: 34.3085 | Accuracy: 0.5659
Epoch 20 | Loss: 33.9981 | Accuracy: 0.5778
Epoch 21 | Loss: 33.9864 | Accuracy: 0.5674
Epoch 22 | Loss: 33.8344 | Accuracy: 0.5822
Epoch 23 | Loss: 34.2674 | Accuracy: 0.59

In [ ]:
print("Best Accuracy:", best_acc)
# it is CSP classifer

Best Accuracy: 0.605925925925926


In [ ]:
X = np.concatenate([t0, t1, t2], axis=0)

y = np.concatenate([
    np.zeros(len(t0)),
    np.ones(len(t1)),
    np.ones(len(t2))*2
]).astype(int)

print(X.shape, y.shape)

(3374, 64, 321) (3374,)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
def bandpass(data, low, high, fs=128, order=4):

    b, a = butter(order, [low/(fs/2), high/(fs/2)], btype='band')

    return filtfilt(b, a, data, axis=-1)

In [ ]:
bands = [
    (4,8),
    (8,12),
    (12,16),
    (16,20),
    (20,24),
    (24,28),
    (28,32)
]

In [ ]:
from scipy.signal import butter, filtfilt

In [ ]:
train_features = []
test_features = []

for low, high in bands:

    X_train_f = bandpass(X_train, low, high)
    X_test_f = bandpass(X_test, low, high)

    csp = CSP(n_components=4, log=True)

    X_train_csp = csp.fit_transform(X_train_f, y_train)
    X_test_csp = csp.transform(X_test_f)

    train_features.append(X_train_csp)
    test_features.append(X_test_csp)

Computing rank from data with rank=None
    Using tolerance 44 (2.2e-16 eps * 64 dim * 3.1e+15  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Estimating class=2 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 42 (2.2e-16 eps * 64 dim * 2.9e+15  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Estimating class=2 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 30 (2.2e-16 eps * 64 dim * 2.1e+15  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels w

In [16]:
import numpy as np
from scipy.signal import butter, filtfilt
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader


# =========================
# 1 Load Data
# =========================

# t0 = np.load("t0.npy")
# t1 = np.load("t1.npy")
# t2 = np.load("t2.npy")

X = np.concatenate([t0, t1, t2], axis=0)

y = np.concatenate([
    np.zeros(len(t0)),
    np.ones(len(t1)),
    np.ones(len(t2)) * 2
]).astype(int)

print("Dataset:", X.shape)


# =========================
# 2 Train Test Split
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


# =========================
# 3 Butterworth 8-30 Hz
# =========================

def bandpass(data, low=8, high=30, fs=128, order=4):

    b, a = butter(order, [low/(fs/2), high/(fs/2)], btype='band')

    return filtfilt(b, a, data, axis=-1)


X_train = bandpass(X_train)
X_test  = bandpass(X_test)


# =========================
# 4 Normalize
# =========================

mean = X_train.mean(axis=(0,2), keepdims=True)
std  = X_train.std(axis=(0,2), keepdims=True)

X_train = (X_train - mean) / (std + 1e-6)
X_test  = (X_test - mean) / (std + 1e-6)


# =========================
# 5 Add CNN Dimension
# =========================

X_train = X_train[:, None, :, :]
X_test  = X_test[:, None, :, :]


# =========================
# 6 PyTorch Dataset
# =========================

train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long)
)

test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.long)
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=64)


# =========================
# 7 Device
# =========================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


# =========================
# 8 ShallowConvNet
# =========================

class ShallowConvNet(nn.Module):

    def __init__(self, n_channels=64, n_samples=321, n_classes=3):

        super().__init__()

        self.temporal = nn.Conv2d(1, 40, (1,25), bias=False)

        self.spatial = nn.Conv2d(40, 40, (n_channels,1), bias=False)

        self.bn = nn.BatchNorm2d(40)

        self.pool = nn.AvgPool2d((1,75), (1,15))

        self.dropout = nn.Dropout(0.5)


        # automatic feature size
        with torch.no_grad():

            dummy = torch.zeros(1,1,n_channels,n_samples)

            x = self.temporal(dummy)
            x = self.spatial(x)
            x = self.bn(x)

            x = torch.square(x)

            x = self.pool(x)

            x = torch.log(torch.clamp(x, min=1e-6))

            x = self.dropout(x)

            self.feature_dim = x.view(1,-1).shape[1]


        self.fc = nn.Linear(self.feature_dim, n_classes)


    def forward(self, x):

        x = self.temporal(x)

        x = self.spatial(x)

        x = self.bn(x)

        x = torch.square(x)

        x = self.pool(x)

        x = torch.log(torch.clamp(x, min=1e-6))

        x = self.dropout(x)

        x = torch.flatten(x,1)

        return self.fc(x)


model = ShallowConvNet().to(device)


# =========================
# 9 Training Setup
# =========================

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.0005,
    weight_decay=1e-4
)


# =========================
# 10 Training
# =========================

epochs = 120
best_acc = 0

save_path = "best_shallowconvnet.pth"

for epoch in range(epochs):

    model.train()

    for xb, yb in train_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        outputs = model(xb)

        loss = criterion(outputs, yb)

        loss.backward()

        optimizer.step()


    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for xb, yb in test_loader:

            xb = xb.to(device)
            yb = yb.to(device)

            outputs = model(xb)

            preds = torch.argmax(outputs, 1)

            total += yb.size(0)

            correct += (preds == yb).sum().item()


    acc = correct / total


    if acc > best_acc:

        best_acc = acc

        torch.save(model.state_dict(), save_path)

        print("Best model saved")


    print("Epoch", epoch+1, "Accuracy:", acc)


print("Best Accuracy:", best_acc)


# =========================
# 11 Load Best Model
# =========================

model.load_state_dict(torch.load(save_path))
model.eval()

print("Best model loaded successfully")

Dataset: (3374, 64, 321)
Device: cuda
Best model saved
Epoch 1 Accuracy: 0.4874074074074074
Best model saved
Epoch 2 Accuracy: 0.5274074074074074
Epoch 3 Accuracy: 0.5155555555555555
Best model saved
Epoch 4 Accuracy: 0.5659259259259259
Epoch 5 Accuracy: 0.5659259259259259
Best model saved
Epoch 6 Accuracy: 0.5777777777777777
Epoch 7 Accuracy: 0.5644444444444444
Best model saved
Epoch 8 Accuracy: 0.5955555555555555
Epoch 9 Accuracy: 0.5511111111111111
Best model saved
Epoch 10 Accuracy: 0.6
Epoch 11 Accuracy: 0.5940740740740741
Epoch 12 Accuracy: 0.5748148148148148
Epoch 13 Accuracy: 0.5703703703703704
Epoch 14 Accuracy: 0.5777777777777777
Epoch 15 Accuracy: 0.5925925925925926
Epoch 16 Accuracy: 0.5748148148148148
Epoch 17 Accuracy: 0.5985185185185186
Epoch 18 Accuracy: 0.6
Epoch 19 Accuracy: 0.5851851851851851
Epoch 20 Accuracy: 0.5792592592592593
Epoch 21 Accuracy: 0.5733333333333334
Epoch 22 Accuracy: 0.5777777777777777
Epoch 23 Accuracy: 0.5940740740740741
Best model saved
Epoch 24

In [8]:
save_path = "/content/drive/MyDrive/shallownet.pth"

In [9]:
torch.save(model.state_dict(), save_path)
print("Model saved to Google Drive")

Model saved to Google Drive


In [11]:
!pip install mne

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 86.2 MB/s eta 0:00:00


In [14]:
import numpy as np
from scipy.signal import butter, filtfilt
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from mne.decoding import CSP


# =========================
# 1 Load Data
# =========================

# t0 = np.load("t0.npy")
# t1 = np.load("t1.npy")
# t2 = np.load("t2.npy")

X = np.concatenate([t1, t2], axis=0)

y = np.concatenate([
    np.ones(len(t1)),
    np.ones(len(t2))*2
]).astype(int)

print("Dataset shape:", X.shape)


# =========================
# 2 Train Test Split
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


# =========================
# 3 Bandpass Filter 8-30 Hz
# =========================

def bandpass(data, low=8, high=30, fs=128, order=4):

    b, a = butter(order, [low/(fs/2), high/(fs/2)], btype='band')

    return filtfilt(b, a, data, axis=-1)


X_train = bandpass(X_train)
X_test  = bandpass(X_test)


# =========================
# 4 Normalize
# =========================

mean = X_train.mean(axis=(0,2), keepdims=True)
std  = X_train.std(axis=(0,2), keepdims=True)

X_train = (X_train - mean) / (std + 1e-6)
X_test  = (X_test - mean) / (std + 1e-6)


# =========================
# 5 Apply CSP
# =========================

n_csp = 32

csp = CSP(
    n_components=n_csp,
    reg=None,
    log=True,
    norm_trace=False
)

X_train_csp = csp.fit_transform(X_train, y_train)
X_test_csp  = csp.transform(X_test)


# reshape for CNN

X_train_csp = X_train_csp[:,:,None]
X_test_csp  = X_test_csp[:,:,None]

X_train_csp = X_train_csp[:,None,:,:]
X_test_csp  = X_test_csp[:,None,:,:]


# =========================
# 6 PyTorch Dataset
# =========================

train_dataset = TensorDataset(
    torch.tensor(X_train_csp, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long)
)

test_dataset = TensorDataset(
    torch.tensor(X_test_csp, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.long)
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=64)


# =========================
# 7 Device
# =========================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)


# =========================
# 8 ShallowConvNet
# =========================

class ShallowConvNet(nn.Module):

    def __init__(self, n_channels=32, n_samples=1, n_classes=3):

        super().__init__()

        self.temporal = nn.Conv2d(1, 40, (1,1), bias=False)

        self.spatial = nn.Conv2d(40, 40, (n_channels,1), bias=False)

        self.bn = nn.BatchNorm2d(40)

        self.pool = nn.AvgPool2d((1,1))

        self.dropout = nn.Dropout(0.5)


        with torch.no_grad():

          self.bn.eval()   # fix batchnorm issue

          dummy = torch.zeros(2,1,n_channels,n_samples)  # batch size 2

          x = self.temporal(dummy)
          x = self.spatial(x)
          x = self.bn(x)

          x = torch.square(x)

          x = self.pool(x)

          x = torch.log(torch.clamp(x,min=1e-6))

          x = self.dropout(x)

          self.feature_dim = x.view(x.size(0),-1).shape[1]

          self.bn.train()


        self.fc = nn.Linear(self.feature_dim, n_classes)


    def forward(self,x):

        x = self.temporal(x)

        x = self.spatial(x)

        x = self.bn(x)

        x = torch.square(x)

        x = self.pool(x)

        x = torch.log(torch.clamp(x,min=1e-6))

        x = self.dropout(x)

        x = torch.flatten(x,1)

        return self.fc(x)


model = ShallowConvNet().to(device)


# =========================
# 9 Training Setup
# =========================

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.0005,
    weight_decay=1e-4
)


# =========================
# 10 Training
# =========================

epochs = 120
best_acc = 0

save_path = "best_csp_shallow.pth"


for epoch in range(epochs):

    model.train()

    for xb, yb in train_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        outputs = model(xb)

        loss = criterion(outputs, yb)

        loss.backward()

        optimizer.step()


    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for xb, yb in test_loader:

            xb = xb.to(device)
            yb = yb.to(device)

            outputs = model(xb)

            preds = torch.argmax(outputs,1)

            total += yb.size(0)

            correct += (preds == yb).sum().item()


    acc = correct / total


    if acc > best_acc:

        best_acc = acc

        torch.save(model.state_dict(), save_path)

        print("Best model saved")


    print("Epoch", epoch+1, "Accuracy:", acc)


print("Best Accuracy:", best_acc)


# =========================
# 11 Load Best Model
# =========================

model.load_state_dict(torch.load(save_path))

model.eval()

print("Best model loaded")

Dataset shape: (1687, 64, 321)
Computing rank from data with rank=None
    Using tolerance 56 (2.2e-16 eps * 64 dim * 3.9e+15  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating class=1 covariance using EMPIRICAL
Done.
Estimating class=2 covariance using EMPIRICAL
Done.
Device: cuda
Best model saved
Epoch 1 Accuracy: 0.3076923076923077
Best model saved
Epoch 2 Accuracy: 0.4289940828402367
Epoch 3 Accuracy: 0.35798816568047337
Epoch 4 Accuracy: 0.4260355029585799
Epoch 5 Accuracy: 0.42011834319526625
Epoch 6 Accuracy: 0.41124260355029585
Best model saved
Epoch 7 Accuracy: 0.47041420118343197
Epoch 8 Accuracy: 0.46153846153846156
Epoch 9 Accuracy: 0.4408284023668639
Epoch 10 Accuracy: 0.4408284023668639
Epoch 11 Accuracy: 0.4556213017751479
Epoch 12 Accuracy: 0.42011834319526625
Epoch 13 Accuracy: 0.4556213017751479
Epoch 14 Accuracy: 0.4467455621301775
Epoch 15 Accuracy: 0.40532